## configurazione del collegamento di databricks con adls attraverso il service principal del registration app (senza connettore)

In [0]:
# CELLA 1 - Configurazione accesso ADLS Gen2
STORAGE_ACCOUNT = "storageaccountautomotive"
TENANT_ID       = "be50fa3b-c833-4bb4-b7ee-9a5b37bcfc2b"
CLIENT_ID       = "e8ca97d1-e3c2-41f4-935d-bb1d3d8add64"
CLIENT_SECRET   = "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"  # sostituisci con il tuo valore

spark.conf.set(f"fs.azure.account.auth.type.{STORAGE_ACCOUNT}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{STORAGE_ACCOUNT}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{STORAGE_ACCOUNT}.dfs.core.windows.net", CLIENT_ID)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{STORAGE_ACCOUNT}.dfs.core.windows.net", CLIENT_SECRET)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{STORAGE_ACCOUNT}.dfs.core.windows.net", f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/token")

print("Configurazione completata!")

auto/raw è la cartella dove la Azure Function deposita i file JSON. Non l'hai creata manualmente — si è creata automaticamente quando la Function ha caricato il primo record. ADLS Gen2 crea le cartelle intermedie da solo quando scrivi un file su un percorso che non esiste ancora

_checkpoints/raw e delta/veicoli non esistono ancora — e va benissimo così. Auto Loader e il writeStream le creano automaticamente al primo avvio:

_checkpoints serve ad Auto Loader per tenere traccia di quali file ha già letto, così se il job si ferma e riparte non rilegge tutto da capo
delta/veicoli è la cartella dove Databricks scrive la Delta Table con i dati processati

In [0]:
# CELLA 2 - Auto Loader
SOURCE_PATH     = "abfss://landing@storageaccountautomotive.dfs.core.windows.net/auto/raw"
CHECKPOINT_PATH = "abfss://landing@storageaccountautomotive.dfs.core.windows.net/auto/_checkpoints/raw"
DELTA_PATH      = "abfss://landing@storageaccountautomotive.dfs.core.windows.net/auto/delta/veicoli"

df_stream = (
    spark.readStream          # leggo in modalità streaming (non batch)
        .format("cloudFiles") # uso Auto Loader
        .option("cloudFiles.format", "json")              # i file sono JSON
        .option("cloudFiles.useNotifications", "false")   # polling sulla cartella
        .option("cloudFiles.includeExistingFiles", "true") # leggi anche i file già presenti
        .load(SOURCE_PATH)    # cartella ADLS da monitorare
)


In questo momento df_stream non fa ancora nulla — è solo una definizione, come una ricetta non ancora cucinata. Auto Loader inizia a leggere solo quando viene "attivato" dalla query.

In [0]:
query = (
    df_stream.writeStream        # scrivo in modalità streaming
        .format("delta")         # formato Delta Lake
        .outputMode("append")    # aggiungi nuove righe, non sovrascrivere
        .option("checkpointLocation", CHECKPOINT_PATH)  # salva lo stato
        .option("mergeSchema", "true")  # se arrivano nuove colonne, adattati
        .trigger(availableNow=True)     # processa tutti i file disponibili ora e fermati
        .start(DELTA_PATH)       # cartella di destinazione
)

query.awaitTermination()
print(f"Completato! Record processati: {query.lastProgress['numInputRows']}")